In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Simulasi eksak dan berbising dengan Qiskit Aer primitives

<details>
<summary><b>Versi paket</b></summary>

Kode di halaman ini dikembangkan menggunakan persyaratan berikut.
Kami menyarankan penggunaan versi ini atau yang lebih baru.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
[Simulasi eksak dengan Qiskit primitives](/guides/simulate-with-qiskit-sdk-primitives) menunjukkan cara menggunakan reference primitives yang disertakan dengan Qiskit untuk melakukan simulasi eksak dari rangkaian kuantum. Prosesor kuantum yang ada saat ini rentan terhadap kesalahan, atau noise, sehingga hasil simulasi eksak belum tentu mencerminkan hasil yang diharapkan saat menjalankan Circuit di hardware nyata. Meski reference primitives di Qiskit tidak mendukung pemodelan noise, [Qiskit Aer](https://qiskit.org/ecosystem/aer/) menyertakan implementasi primitives yang mendukung pemodelan noise. Qiskit Aer adalah simulator Circuit kuantum berkinerja tinggi yang bisa kamu gunakan sebagai pengganti reference primitives untuk performa lebih baik dan fitur lebih banyak. Ini merupakan bagian dari [Qiskit Ecosystem](https://qiskit.github.io/ecosystem/). Dalam artikel ini, kami menunjukkan penggunaan Qiskit Aer primitives untuk simulasi eksak dan berbising.

> **Note:** - `qiskit-aer` v0.14 atau lebih baru diperlukan.
> - Meski Qiskit Aer primitives mengimplementasikan antarmuka primitive, mereka tidak menyediakan opsi yang sama dengan Qiskit Runtime primitives. Resilience level, misalnya, tidak tersedia dengan Qiskit Aer primitives.
> - Lihat [dokumentasi AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator) untuk detail tentang opsi metode simulasi yang didukung Aer.

Untuk menjelajahi simulasi eksak dan berbising, buat contoh Circuit pada delapan Qubit:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

Circuit ini mengandung parameter untuk mewakili sudut rotasi Gate $R_y$ dan $R_z$. Saat mensimulasikan Circuit ini, kita perlu menentukan nilai eksplisit untuk parameter-parameter tersebut. Di sel berikutnya, kita tentukan beberapa nilai untuk parameter ini dan menggunakan Estimator primitive dari Qiskit Aer untuk menghitung nilai harapan eksak dari observable $ZZ \cdots Z$.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

Sekarang, mari inisialisasi model noise yang menyertakan depolarizing error sebesar 2% pada setiap Gate CX. Dalam praktiknya, kesalahan yang muncul dari Gate dua Qubit, yaitu Gate CX di sini, merupakan sumber kesalahan utama saat menjalankan Circuit. Lihat [Membangun model noise](./build-noise-models) untuk gambaran umum pembuatan model noise di Qiskit Aer.

Di sel berikutnya, kita membangun Estimator yang menggabungkan model noise ini dan menggunakannya untuk menghitung nilai harapan dari observable.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

Seperti yang kamu lihat, nilai harapan dengan adanya noise cukup jauh dari nilai yang benar. Dalam praktiknya, kamu bisa menggunakan berbagai teknik mitigasi kesalahan untuk melawan efek noise, namun pembahasan teknik-teknik ini berada di luar cakupan artikel ini.

Untuk mendapat gambaran kasar tentang bagaimana noise mempengaruhi hasil akhir, perhatikan model noise kita yang menambahkan depolarizing error sebesar 2% ke setiap Gate CX. Depolarizing error dengan probabilitas $p$ didefinisikan sebagai saluran kuantum $E$ yang memiliki aksi berikut pada matriks densitas $\rho$:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

di mana $n$ adalah jumlah Qubit, dalam hal ini 2. Artinya, dengan probabilitas $p$, keadaan digantikan dengan keadaan tercampur sempurna, dan keadaan dipertahankan dengan probabilitas $1 - p$. Setelah $m$ aplikasi depolarizing channel, probabilitas keadaan yang dipertahankan adalah $(1 - p)^m$. Oleh karena itu, kita mengharapkan probabilitas mempertahankan keadaan yang benar di akhir simulasi turun secara eksponensial seiring bertambahnya jumlah Gate CX dalam Circuit kita.

Mari hitung jumlah Gate CX dalam Circuit kita dan hitung $(1 - p)^m$. Kita panggil `count_ops` untuk mendapatkan kamus yang memetakan nama Gate ke jumlahnya, lalu ambil entri untuk Gate CX.